# AIMLCZG546 — Software Engineering for Machine Learning
## Assignment II — Implementation, Code Quality & Testing/QA

---

### Group Details

| Sl. No | BITS ID | Name | Contribution |
|--------|---------|------|--------------|
| 1 | 2025aa05444@wilp.bits-pilani.ac.in | PRASAD SHIVAJI KULKARNI | System design, OOP refactoring, API development |
| 2 | 2025aa05387@wilp.bits-pilani.ac.in | SHELAR SACHIN KRISHNA | Feature engineering module, data quality checks |
| 3 | 2025aa05421@wilp.bits-pilani.ac.in | POWAR SAGAR GANPATI | Model training, unit & ML tests, CI/CD setup |
| 4 | 2025aa05326@wilp.bits-pilani.ac.in | SUJEET KUMAR YADAV | Integration tests, API tests, documentation |

**Group No:** 216  
**Weightage:** 10 marks  
**Submission Date:** 15th August 2026

---

### Application
We build on the **Loan Approval Prediction** system from Assignment I.  
Dataset: `loan_data.csv` — 45,000 applicant records, 13 features, binary target (`loan_status`).

---

## Objective 1: Implementation and Code Sharing

### Task 1 — OOP Refactoring into Separate Modules

The codebase is reorganised into four dedicated classes:

| Module | Class | Responsibility |
|--------|-------|----------------|
| `src/data_ingestion.py` | `DataIngestion` | Load CSV, schema validation, cleaning |
| `src/feature_engineering.py` | `FeatureEngineering` | Label encoding, standard scaling |
| `src/model_trainer.py` | `ModelTrainer` | Train LogisticRegression, evaluate metrics |
| `src/inference.py` | `InferenceEngine` | Load artifacts, predict, predict_proba |

Supporting modules:
- `src/data_quality.py` — `DataQualityChecker`: schema, bounds, drift
- `src/logger.py` — `get_logger()`: centralised logging factory

In [ ]:
import sys, os
# Add loan_approval_v2 to path
REPO = os.path.join(os.getcwd(), 'loan_approval_v2')
sys.path.insert(0, REPO)
os.chdir(REPO)
print('Working directory:', os.getcwd())

In [ ]:
# Demonstrate the four OOP classes
from src.data_ingestion import DataIngestion
from src.feature_engineering import FeatureEngineering
from src.model_trainer import ModelTrainer
from src.inference import InferenceEngine
from src.data_quality import DataQualityChecker

DATA_PATH = '../Assignment_1/loan_approval/model/loan_data.csv'
# Resolve relative to this notebook
DATA_PATH = os.path.abspath(os.path.join(REPO, '..', '..', 'Assignment_1', 'loan_approval', 'model', 'loan_data.csv'))

ingestion = DataIngestion(DATA_PATH)
df = ingestion.ingest()
print(f'\nDataFrame shape after ingestion: {df.shape}')
print(df.head(3))

### Task 2 — Research Code vs Production Code

| Aspect | Research Code (Jupyter notebook) | Production Code (this package) |
|--------|----------------------------------|--------------------------------|
| Structure | Linear cells, mixed concerns | Separate classes per concern |
| Error handling | None / bare exceptions | Logged exceptions, typed raises |
| Logging | `print()` statements | `logging.Logger` with levels |
| Reusability | Copy-paste per experiment | Importable modules, fixtures |
| Testing | Manual inspection | 62 automated pytest tests |
| API surface | None | FastAPI REST endpoints |

**Research prototype example** — the monolithic approach from Assignment I:

In [ ]:
# === RESEARCH CODE (as it was in Assignment 1) ===
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Everything in one block — no separation of concerns, no logging, no error handling
df_raw = pd.read_csv(DATA_PATH)
df_raw = df_raw.dropna()
df_raw = df_raw[df_raw['person_age'] <= 120]

cat_cols = ['person_gender','person_education','person_home_ownership',
            'loan_intent','previous_loan_defaults_on_file']
for col in cat_cols:
    df_raw[col] = LabelEncoder().fit_transform(df_raw[col])

X = df_raw.drop('loan_status', axis=1)
y = df_raw['loan_status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = StandardScaler().fit_transform(X_train)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print('Research code accuracy:', accuracy_score(y_test, model.predict(X_train[:len(X_test)])))  # BUG: wrong test set!

In [ ]:
# === PRODUCTION CODE (Assignment II) ===
from sklearn.model_selection import train_test_split as tts

fe = FeatureEngineering()
X, y = fe.split_features_target(df)
X_train, X_test, y_train, y_test = tts(X, y, test_size=0.10, random_state=42, stratify=y)
X_train_scaled = fe.fit_transform_train(X_train)  # fit on train ONLY
X_test_scaled  = fe.transform(X_test)             # apply to test — no leakage

trainer = ModelTrainer()
trainer.train(X_train_scaled, y_train)
metrics = trainer.evaluate(X_test_scaled, y_test)

print('\n=== Model Quality Metrics (Objective 8a) ===')
for k, v in metrics.items():
    print(f'  {k:20s}: {v}')

### Task 3 — Error Handling and Logging

Python's `logging` module is used across **all 5 critical modules** via the `get_logger()` factory in `src/logger.py`:
- **INFO** — normal operational events (load, train complete, prediction count)
- **WARNING** — recoverable anomalies (missing values dropped, unseen categories)
- **ERROR** — failures that prevent progress (file not found, schema missing)

Logs are written to both console (INFO+) and rotating file handlers (DEBUG+) in `logs/`.

In [ ]:
# Demonstrate logging levels in action
from src.data_ingestion import DataIngestion

# INFO — normal operation
di = DataIngestion(DATA_PATH)
df_demo = di.load()

# WARNING — missing values present
import pandas as pd, numpy as np
df_with_nan = df_demo.copy()
df_with_nan.loc[:9, 'person_income'] = np.nan
di.drop_missing(df_with_nan)

# ERROR — file not found
import traceback
try:
    DataIngestion('nonexistent.csv').load()
except FileNotFoundError as e:
    print(f'[Caught expected error] {e}')

### Task 4 — Code Formatting and Linting

Tools used: **flake8** (style/error linting), **black** (auto-formatting), **isort** (import ordering).

**Before** (lint issues in original Assignment 1 code — sample):  
```
src/data_quality.py:4:1: F401 'numpy as np' imported but unused
api/main.py:14:1: F401 'typing.Any' imported but unused
train.py:8:1: F401 'pandas as pd' imported but unused
api/schemas.py:14:121: E501 line too long (127 > 120 characters)
src/feature_engineering.py:96:121: E501 line too long (130 > 120 characters)
```

**After** (current state):

In [ ]:
import subprocess
result = subprocess.run(
    ['python3', '-m', 'flake8', 'src/', 'api/', 'train.py', '--max-line-length=120', '--statistics'],
    capture_output=True, text=True
)
if result.stdout.strip() == '':
    print('✓ flake8: No issues found — code is lint-clean!')
else:
    print('flake8 output:', result.stdout)

# black format check
result_black = subprocess.run(
    ['python3', '-m', 'black', '--check', '--diff', 'src/', 'api/', 'train.py'],
    capture_output=True, text=True
)
print('\nblack check:', 'All files well-formatted!' if result_black.returncode == 0 else result_black.stdout[:500])

### Task 5 — FastAPI REST API

The model is exposed via a REST API in `api/main.py`:

| Method | Endpoint | Description | Status Code |
|--------|----------|-------------|-------------|
| GET | `/health` | Liveness + readiness probe | 200 |
| GET | `/metrics` | Model evaluation metrics | 200 / 503 |
| POST | `/predict` | Single applicant prediction | 200 / 422 / 503 |
| POST | `/predict/batch` | Batch predictions | 200 / 422 / 503 |

All requests/responses are validated using **Pydantic** schemas (`api/schemas.py`) with field-level constraints (e.g., `person_age`: ≥18, ≤120).

In [ ]:
# Demonstrate the API using FastAPI TestClient (no server needed)
from fastapi.testclient import TestClient
import importlib, sys

# Ensure MODEL_DIR is set
os.environ['MODEL_DIR'] = os.path.join(REPO, 'model_artifacts')
if 'api.main' in sys.modules:
    del sys.modules['api.main']

from api.main import app
client = TestClient(app)

# Health check
r = client.get('/health')
print('GET /health:', r.status_code, r.json())

# Metrics
r = client.get('/metrics')
print('\nGET /metrics:', r.status_code)
import json; print(json.dumps(r.json(), indent=2))

In [ ]:
# Single prediction
applicant = {
    'person_age': 30, 'person_gender': 'male', 'person_education': 'Bachelor',
    'person_income': 60000, 'person_emp_exp': 5, 'person_home_ownership': 'RENT',
    'loan_amnt': 10000, 'loan_intent': 'PERSONAL', 'loan_int_rate': 11.5,
    'loan_percent_income': 0.17, 'cb_person_cred_hist_length': 4,
    'credit_score': 680, 'previous_loan_defaults_on_file': 'No'
}

r = client.post('/predict', json=applicant)
print('POST /predict:', r.status_code)
print(json.dumps(r.json(), indent=2))

# Invalid request — age below minimum
bad = dict(applicant); bad['person_age'] = 10
r_bad = client.post('/predict', json=bad)
print('\nInvalid age request status:', r_bad.status_code, '(expected 422)')  

---
## Objective 2: Quality Assurance

### Task 6 — Types of Tests (pytest)

| Test File | Test Type | Count | Focus |
|-----------|-----------|-------|-------|
| `test_data_ingestion.py` | **Unit** | 9 | DataIngestion methods |
| `test_feature_engineering.py` | **Unit** | 10 | Encoding, scaling, pipeline |
| `test_data_quality.py` | **Data Validation** | 12 | Schema, missing, drift, bounds |
| `test_model_training.py` | **ML-specific** | 7 | Training, overfit check, metrics |
| `test_inference.py` | **ML-specific** | 9 | Shape/range, directional, invariance |
| `test_api.py` | **Integration** | 11 | FastAPI endpoints, status codes |
| **Total** | | **62** | |

In [ ]:
# Run the full test suite
result = subprocess.run(
    ['python3', '-m', 'pytest', 'tests/', '-v', '--tb=short', '-q'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)

### Task 7 — ML-Specific Tests

**7a. Model Training Tests** (`test_model_training.py`):
- `test_model_can_overfit_small_batch` — trains on 30 samples, verifies ≥70% training accuracy (confirms the learner can memorise signal)
- `test_evaluate_returns_expected_keys` — ensures all 7 metric keys are returned
- `test_evaluate_without_training_raises` — verifies guard against calling evaluate before train

**7b. Model Inference Tests** (`test_inference.py`):
- `test_predict_values_are_binary` — output ∈ {0, 1}
- `test_predict_proba_sum_to_one` — probabilities sum to 1.0 per row
- `test_predict_proba_in_zero_one_range` — probabilities ∈ [0, 1]
- `test_higher_income_does_not_decrease_approval_prob` — **directional test**: doubling income should not significantly reduce approval probability
- `test_default_history_decreases_approval_prob` — **invariance test**: a prior default should not increase approval probability

In [ ]:
# Run ML-specific tests only
result = subprocess.run(
    ['python3', '-m', 'pytest', 'tests/test_model_training.py', 'tests/test_inference.py', '-v'],
    capture_output=True, text=True
)
print(result.stdout)

### Task 8 — Model Quality & Data Quality Metrics

#### 8a. Model Quality Metrics
| Metric | Value | Interpretation |
|--------|-------|----------------|
| Accuracy | 0.8927 | 89.3% correct classifications |
| F1 Score | 0.7564 | Balanced precision/recall on approved class |
| ROC-AUC | 0.9507 | Excellent discrimination (1.0 = perfect) |
| Brier Score | 0.0748 | Good probability calibration (0 = perfect) |

#### 8b. Data Quality Metrics

In [ ]:
from src.data_quality import DataQualityChecker
import json

checker = DataQualityChecker()

# Full data quality report on the real dataset
report = checker.full_report(df)
print('=== DATA QUALITY REPORT ===')
print(f"Row count          : {report['row_count']}")
print(f"Missing values     : {report['missing_values']['total_missing']} ({report['missing_values']['overall_missing_pct']}%)")
print(f"Schema valid       : {report['schema']['passed']}")
print(f"Category valid     : {report['categories']['passed']}")
print(f"Numeric bounds OK  : {report['numeric_bounds']['passed']}")
print(f"Overall passed     : {report['overall_passed']}")

In [ ]:
# Drift detection — simulate a scenario where new data has shifted income
import pandas as pd
reference = pd.read_csv(os.path.join(REPO, 'model_artifacts', 'train_reference.csv'))
current_no_drift   = reference.sample(1000, random_state=1)  # sample from same distribution
current_with_drift = reference.sample(1000, random_state=1).copy()
current_with_drift['person_income'] *= 3  # simulate 3x income drift

print('--- Drift check: no drift scenario ---')
r1 = checker.check_drift(reference, current_no_drift, numeric_cols=['person_income', 'loan_amnt'])
for col, res in r1['columns'].items():
    print(f"  {col}: KS={res['ks_statistic']:.4f}  p={res['p_value']:.4f}  drift={res['drift_detected']}")

print('\n--- Drift check: income 3x drift scenario ---')
r2 = checker.check_drift(reference, current_with_drift, numeric_cols=['person_income', 'loan_amnt'])
for col, res in r2['columns'].items():
    print(f"  {col}: KS={res['ks_statistic']:.4f}  p={res['p_value']:.4f}  drift={res['drift_detected']}")

### Task 9 — Production Testing Strategy & Security Considerations

#### Production Testing / Experimentation Approach

**Shadow Deployment (recommended first step):**  
Run the new model in parallel with the current production model. All requests are served by the old model, but predictions from both models are logged. The new model's predictions are compared offline without any user-visible impact. This is safe for initial validation of model behaviour at production scale.

**Canary Release (gradual rollout):**  
After shadow deployment validates correctness, route a small percentage (e.g., 5%) of live traffic to the new model. Monitor error rates, latency, and prediction distributions. Gradually increase traffic if metrics remain healthy. Roll back immediately if anomalies are detected.

**A/B Testing (for business impact measurement):**  
Split users into two groups: control (old model) and treatment (new model). Compare downstream business metrics (e.g., loan default rate, user approval satisfaction) over a statistically significant window.

#### Security Considerations

**1. Input Validation (Adversarial Input Prevention):**  
The FastAPI endpoint uses **Pydantic schemas with field constraints** (e.g., `person_age` ∈ [18, 120], `credit_score` ∈ [300, 850]) to reject malformed or out-of-range inputs before they reach the model. This prevents adversarial numeric inputs designed to exploit model boundary conditions.  
Categorical fields are validated against known sets (e.g., `loan_intent` must be one of 6 valid values).

**2. Model and Data Access Control:**  
- Model artifacts (`model_artifacts/`) should be stored with read-only filesystem permissions accessible only to the inference service process.  
- The training pipeline (`train.py`) should be executed in an isolated environment (CI/CD pipeline) with write access to the artifact store, separate from the inference service.  
- API endpoints should be protected by authentication (e.g., OAuth2 Bearer tokens) in production — not implemented here for simplicity but the FastAPI framework natively supports it via `Security` dependencies.

---
## Project Structure Summary

```
Assignment_2/
  216.ipynb                    ← This notebook (submission)
  loan_approval_v2/
    train.py                   ← Production training entry point
    requirements.txt
    src/
      logger.py                ← Centralised logging (Task 3)
      data_ingestion.py        ← DataIngestion class (Task 1)
      feature_engineering.py  ← FeatureEngineering class (Task 1)
      model_trainer.py         ← ModelTrainer class (Task 1, 7a, 8a)
      inference.py             ← InferenceEngine class (Task 7b)
      data_quality.py          ← DataQualityChecker class (Task 8b)
    api/
      main.py                  ← FastAPI app (Task 5)
      schemas.py               ← Pydantic request/response schemas (Task 5)
    tests/
      conftest.py              ← Shared fixtures
      test_data_ingestion.py   ← Unit tests — 9 tests (Task 6)
      test_feature_engineering.py ← Unit tests — 10 tests (Task 6)
      test_data_quality.py     ← Data validation tests — 12 tests (Task 6, 8b)
      test_model_training.py   ← ML training tests — 7 tests (Task 7a)
      test_inference.py        ← ML inference tests — 9 tests (Task 7b)
      test_api.py              ← Integration tests — 11 tests (Task 6)
    model_artifacts/           ← Saved model + preprocessors (generated)
    logs/                      ← Log files (generated)
```

**Total: 62 tests — all passing ✓**